In [ ]:
!pip install uv pyngrok

In [ ]:
from pyngrok import ngrok
import os

# Điền token của bạn vào đây
NGROK_AUTH_TOKEN = ""
os.system(f"ngrok config add-authtoken {NGROK_AUTH_TOKEN}")

# Mở port 8000 (port mặc định của vLLM)
public_url = ngrok.connect(8000).public_url

print("="*60)
print(f"👉 URL ĐỂ GỌI TỪ VS CODE (LOCAL): {public_url}/v1")
print("="*60)

In [ ]:
import os

bash_script_content = """#!/bin/bash
set -e

MODEL="hoangchihien3011/VietMind"
PORT=8000
MAX_MODEL_LEN=64000


echo ">>> Remove old venv if exists..."

rm -rf .venv


echo ">>> Update system packages..."

apt-get update -y
apt-get install -y build-essential


echo ">>> Create venv..."

uv venv .venv


echo ">>> Install pip..."

uv pip install \\
--python .venv/bin/python \\
pip setuptools wheel


echo ">>> Install PyTorch CUDA 13..."

uv pip install \\
--python .venv/bin/python \\
torch torchvision torchaudio \\
--index-url https://download.pytorch.org/whl/cu130


echo ">>> Check CUDA..."

.venv/bin/python - <<EOF
import torch

print("==============================")
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("==============================")
EOF



echo ">>> Install build tools..."

uv pip install \\
--python .venv/bin/python \\
ninja cmake packaging


echo ">>> Verify ninja..."

.venv/bin/ninja --version



echo ">>> Install vLLM..."

uv pip install \\
--python .venv/bin/python \\
openai vllm \\
--extra-index-url https://wheels.vllm.ai/nightly



echo ">>> Clean old cache..."

rm -rf ~/.cache/flashinfer
rm -rf ~/.cache/vllm



echo ">>> Start vLLM..."

# export HF_TOKEN=YOUR_TOKEN_HERE

source .venv/bin/activate

vllm serve $MODEL \\
--port $PORT \\
--gpu-memory-utilization 0.9 \\
--max-model-len $MAX_MODEL_LEN \\
--trust-remote-code
"""


output_filename = "run_pipeline.sh"

with open(output_filename, "w", encoding="utf-8") as f:
    f.write(bash_script_content)

os.chmod(output_filename, 0o755)

print(f"✅ Created {output_filename}")

In [ ]:
!./run_pipeline.sh